In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import importlib

import src.utils.state as state
import src.utils.plottings as plottings
import src.utils.scenarioCalculator as scenarioCalculator
import src.utils.dataConverter as dataConverter
from src.utils.dataConverter import convert_df_age_structured_cases, create_case_matrix
import src.utils.models as models
from src.utils.scenarioCalculator import ScenarioCalculator
from src.utils.scenarios import Scenarios

importlib.reload(plottings)
importlib.reload(state)
importlib.reload(scenarioCalculator)
importlib.reload(dataConverter)
importlib.reload(models)

from src.utils.models import Scenario

## 1. Reading data

In [ ]:
df_cases = pd.read_excel("../resources/weekly_varicella_hun.xlsx", sheet_name="weekly_cases")
df_age_structured_cases = pd.read_excel("../resources/age_structured_vzv_hun.xlsx", sheet_name="varicella")
df_susceptibles = pd.read_excel("../resources/number_of_susceptibles_hun.xlsx", sheet_name="s0")
df_population = pd.read_excel("../resources/age_structured_population.xlsx", sheet_name="population")
df_births = pd.read_excel("../resources/births_hun.xlsx", sheet_name="births")
df_daily_births = pd.read_excel("../resources/births_hun.xlsx", sheet_name="daily", header=None)
df_deaths = pd.read_excel("../resources/age_structured_deaths_hun.xlsx", sheet_name="deaths")
df_death_proportions = pd.read_excel("../resources/age_structured_deaths_hun.xlsx", sheet_name="proportions")
df_vaccines = pd.read_excel("../resources/vaccine_coverage_hun.xlsx", sheet_name="vaccines")
df_contacts = pd.read_excel("../resources/contact_mtx_hun.xlsx", sheet_name="contacts", index_col=0)

## 2. Convert dates and set indices

### 2.1 Varicella data

#### 2.1.1 Weekly cases

In [ ]:
weekly_cases = dataConverter.convert_df_cases_to_weekly_cases(df_cases)
state.set_weekly_cases(weekly_cases)

# fill missing values
full_index = dataConverter.create_full_index_date_range(weekly_cases.index.max())
state.set_full_index(full_index)

state.set_weekly_cases_related_values()

#### 2.1.2 Age-structured data

In [ ]:
state.set_age_groups_and_nr_age_groups(df_age_structured_cases.iloc[:, 0].astype(str).tolist())
state.set_years(df_age_structured_cases.columns[1:].astype(int).tolist())

annual_cases = convert_df_age_structured_cases(df_age_structured_cases)
state.set_annual_cases(annual_cases)

In [ ]:
case_matrix = create_case_matrix(annual_cases, state.WEEKLY_CASES_FILLED, state.WEEKLY_CASES_FULL)
state.set_cases_matrix_and_global_min_max(case_matrix)

In [ ]:
state.set_age_structured_data_mask(state.WEEKLY_CASES_FILLED.index.year <= max(state.YEARS))
state.set_age_structured_data_index(state.WEEKLY_CASES_FILLED.loc[state.AGE_STRUCTURED_DATA_MASK].index)
i = case_matrix
state.set_nr_timesteps(len(state.WEEKLY_CASES_FILLED[state.AGE_STRUCTURED_DATA_MASK]))

### 2.2 Birth and death data; vaccination coverage

In [ ]:
state.set_weekly_index_related_values(state.WEEKLY_CASES_FILLED[state.AGE_STRUCTURED_DATA_MASK].index)

annual_deaths = dataConverter.convert_annual_deaths(df_deaths)
death_matrix = dataConverter.create_weekly_death_matrix(annual_deaths, state.WEEKLY_INDEX)
#death_matrix = dataConverter.create_weekly_death_matrix(annual_deaths, state.WEEKLY_CASES_FULL)
#state.set_death_matrix(death_matrix)

deaths_prop_matrix = dataConverter.convert_weekly_death_prop_data_to_death_prop_mtx(df_death_proportions)
state.set_death_matrix(deaths_prop_matrix)

In [ ]:
#birth_mtx = dataConverter.convert_annual_birth_data_to_birth_mtx(df_births)
birth_mtx = dataConverter.convert_daily_birth_data_to_birth_mtx(df_daily_births)
state.set_birth_matrix(birth_mtx)
weekly_v1 = dataConverter.convert_annual_v1_data_to_weekly_v1(df_vaccines)
state.set_weekly_v1(weekly_v1)
weekly_v2 = dataConverter.convert_annual_v2_data_to_weekly_v2(df_vaccines)
state.set_weekly_v2(weekly_v2)

### 2.3 Initial values

#### 2.3.1 Number of susceptibles

In [ ]:
S0_vector = dataConverter.convert_data_to_s0(df_susceptibles)
state.set_s0_vector(S0_vector)

#### 2.3.2 Population size

In [ ]:
N0_vector = dataConverter.convert_data_to_n0(df_population)
state.set_n0_vector(N0_vector)

### 2.4 Contact matrix

In [ ]:
contacts0 = dataConverter.convert_data_to_contact_matrix(df_contacts)
state.set_contacts0(contacts0)

## 3. Calculate number of susceptibles

In [ ]:
ALL_SCENARIOS: list[Scenario] = []

In [ ]:
state.set_weekly_v1_age_structured()
state.set_weekly_v2_age_structured()

ScenarioCalculator.calculate_number_of_susceptible_cases_in_baseline()
ALL_SCENARIOS.append(state.BASELINE_SCENARIO)

## 4. Estimate R_t using renewal equation model

In [ ]:
ScenarioCalculator.calculate_remainders()
plottings.plot_vector_with_index(state.RHO, state.AGE_STRUCTURED_DATA_INDEX, "Rho", "Date", "rho")

## 5. Plot results for R_t

In [ ]:
plottings.plot_remainders()

## 6. Plot results for the number of incidences

In [ ]:
ScenarioCalculator.calculate_number_of_incidences_based_on_remainders()

#plottings.plot_model_actual_scat_comparison(state.I_BASELINE)
plottings.plot_model_actual_scat_comparison(state.BASELINE_SCENARIO)

In [ ]:
annual_infections_scen0 = state.I_BASELINE.groupby(state.I_BASELINE.index.year).sum()
annual_infections_scen0.index = pd.to_datetime(annual_infections_scen0.index, format="%Y")
state.set_i_baseline_annual(annual_infections_scen0)

plottings.plot_annual_cases(annual_infections_scen0)

In [ ]:
plottings.plot_model_results_age_range_in_range(state.BASELINE_SCENARIO.I, 0, 4)
plottings.plot_model_results_age_range_in_range(state.BASELINE_SCENARIO.I, 5, 9)
plottings.plot_model_results_age_range_in_range(state.BASELINE_SCENARIO.I, 10, 12)

In [ ]:
plottings.plot_scat_age_range_in_range(state.BASELINE_SCENARIO.I, state.AGE_STRUCTURED_DATA_INDEX, 0, 4, state.CASES_MATRIX, state.AGE_STRUCTURED_DATA_MASK)
plottings.plot_scat_age_range_in_range(state.BASELINE_SCENARIO.I, state.AGE_STRUCTURED_DATA_INDEX, 5, 9, state.CASES_MATRIX, state.AGE_STRUCTURED_DATA_MASK)
plottings.plot_scat_age_range_in_range(state.BASELINE_SCENARIO.I, state.AGE_STRUCTURED_DATA_INDEX, 10, 12, state.CASES_MATRIX, state.AGE_STRUCTURED_DATA_MASK)

## 7. Incidences without vaccination

In [ ]:
scenario_no_vacc = scenarioCalculator.calculate_number_of_cases_for_scenario(Scenarios.NO_VACCINATION, 0, 0)
ALL_SCENARIOS.append(scenario_no_vacc)

In [ ]:
plottings.plot_model_actual_scat_comparison(scenario_no_vacc)
plottings.plot_model_results_age_range_in_range(scenario_no_vacc.I,0,4)
plottings.plot_model_results_age_range_in_range(scenario_no_vacc.I,5,9)
plottings.plot_model_results_age_range_in_range(scenario_no_vacc.I,10,12)

In [ ]:
plottings.plot_compare_scenario_to_actual_case_for_age_groups(scenario_no_vacc.I,0,4,scenario_no_vacc.name)

In [ ]:
plottings.plot_compare_scenario_to_actual_case(scenario_no_vacc.weekly_i_series, scenario_no_vacc.name)

## 8. Vaccination level = 0.75

In [ ]:
scenario_vacc_level_75: Scenario = scenarioCalculator.calculate_number_of_cases_for_scenario(Scenarios.VACC_LEVEL_75, 0.75, 0)
ALL_SCENARIOS.append(scenario_vacc_level_75)

In [ ]:
plottings.plot_model_actual_scat_comparison(scenario_vacc_level_75)
plottings.plot_model_results_age_range_in_range(scenario_vacc_level_75.I,0,4)
plottings.plot_model_results_age_range_in_range(scenario_vacc_level_75.I,5,9)
plottings.plot_model_results_age_range_in_range(scenario_vacc_level_75.I,10,12)

In [ ]:
plottings.plot_compare_scenario_to_actual_case(scenario_vacc_level_75.weekly_i_series, scenario_vacc_level_75.name)

## 9. Vaccination level = 0.5

In [ ]:
scenario_vacc_level_50: Scenario = scenarioCalculator.calculate_number_of_cases_for_scenario(Scenarios.VACC_LEVEL_50, 0.50, 0)
ALL_SCENARIOS.append(scenario_vacc_level_50)

In [ ]:
plottings.plot_model_actual_scat_comparison(scenario_vacc_level_50)
plottings.plot_model_results_age_range_in_range(scenario_vacc_level_50.I,0,4)
plottings.plot_model_results_age_range_in_range(scenario_vacc_level_50.I,5,9)
plottings.plot_model_results_age_range_in_range(scenario_vacc_level_50.I,10,12)

In [ ]:
plottings.plot_compare_scenario_to_actual_case(scenario_vacc_level_50.weekly_i_series, scenario_vacc_level_50.name)

## 10. Vaccination level = 0.25

In [ ]:
scenario_vacc_level_25: Scenario = scenarioCalculator.calculate_number_of_cases_for_scenario(Scenarios.VACC_LEVEL_25, 0.25, 0)
ALL_SCENARIOS.append(scenario_vacc_level_25)

In [ ]:
plottings.plot_model_actual_scat_comparison(scenario_vacc_level_25)
plottings.plot_model_results_age_range_in_range(scenario_vacc_level_25.I, 0, 4)
plottings.plot_model_results_age_range_in_range(scenario_vacc_level_25.I, 5, 9)
plottings.plot_model_results_age_range_in_range(scenario_vacc_level_25.I, 10, 12)

In [ ]:
plottings.plot_compare_scenario_to_actual_case(scenario_vacc_level_25.weekly_i_series, scenario_vacc_level_25.name)

## Starting vaccination earlier

## 11. Start vaccination 1 year earlier (2018.09.01)

In [ ]:
scenario_starting_0_year_minus: Scenario = scenarioCalculator.calculate_number_of_cases_for_scenario(Scenarios.STARTING_1_YEAR_MINUS, 1, 0)

In [ ]:
plottings.plot_model_actual_scat_comparison(scenario_starting_0_year_minus)
plottings.plot_model_results_age_range_in_range(scenario_starting_0_year_minus.I, 0, 4)
plottings.plot_model_results_age_range_in_range(scenario_starting_0_year_minus.I, 5, 9)
plottings.plot_model_results_age_range_in_range(scenario_starting_0_year_minus.I, 10, 12)

In [ ]:
scenario_1_year_minus: Scenario = scenarioCalculator.calculate_number_of_cases_for_scenario(Scenarios.STARTING_1_YEAR_MINUS, 1, -1)
ALL_SCENARIOS.append(scenario_1_year_minus)

In [ ]:
plottings.plot_model_actual_scat_comparison(scenario_1_year_minus)
plottings.plot_model_results_age_range_in_range(scenario_1_year_minus.I, 0, 4)
plottings.plot_model_results_age_range_in_range(scenario_1_year_minus.I, 5, 9)
plottings.plot_model_results_age_range_in_range(scenario_1_year_minus.I, 10, 12)

In [ ]:
plottings.plot_compare_scenario_to_actual_case(scenario_1_year_minus.weekly_i_series, scenario_starting_0_year_minus.name)
plottings.plot_compare_scenario_to_actual_case_for_age_groups(scenario_1_year_minus.I, 0, 4, scenario_starting_0_year_minus.name)
plottings.plot_compare_scenario_to_actual_case_for_age_groups(scenario_1_year_minus.I, 5, 9, scenario_starting_0_year_minus.name)
plottings.plot_compare_scenario_to_actual_case_for_age_groups(scenario_1_year_minus.I, 10, 12, scenario_starting_0_year_minus.name)

## 12. Start vaccination 2 years earlier (2017.09.01)

In [ ]:
scenario_2_years_minus: Scenario = scenarioCalculator.calculate_number_of_cases_for_scenario(Scenarios.STARTING_2_YEAR_MINUS, 1, -2)
ALL_SCENARIOS.append(scenario_2_years_minus)

In [ ]:
plottings.plot_model_actual_scat_comparison(scenario_2_years_minus)
plottings.plot_model_results_age_range_in_range(scenario_2_years_minus.I, 0, 4)
plottings.plot_model_results_age_range_in_range(scenario_2_years_minus.I, 5, 9)
plottings.plot_model_results_age_range_in_range(scenario_2_years_minus.I, 10, 12)

In [ ]:
plottings.plot_compare_scenario_to_actual_case(scenario_2_years_minus.weekly_i_series, scenario_2_years_minus.name)
plottings.plot_compare_scenario_to_actual_case_for_age_groups(scenario_2_years_minus.I, 0, 4, scenario_2_years_minus.name)
plottings.plot_compare_scenario_to_actual_case_for_age_groups(scenario_2_years_minus.I, 5, 9, scenario_2_years_minus.name)
plottings.plot_compare_scenario_to_actual_case_for_age_groups(scenario_2_years_minus.I, 10, 12, scenario_2_years_minus.name)

## 13. Start vaccination 3 years earlier (2016.09.01)

In [ ]:
scenario_3_years_minus: Scenario = scenarioCalculator.calculate_number_of_cases_for_scenario(Scenarios.STARTING_3_YEAR_MINUS, 1, -3)
ALL_SCENARIOS.append(scenario_3_years_minus)

In [ ]:
plottings.plot_model_actual_scat_comparison(scenario_3_years_minus)
plottings.plot_model_results_age_range_in_range(scenario_3_years_minus.I, 0, 4)
plottings.plot_model_results_age_range_in_range(scenario_3_years_minus.I, 5, 9)
plottings.plot_model_results_age_range_in_range(scenario_3_years_minus.I, 10, 12)

In [ ]:
plottings.plot_compare_scenario_to_actual_case(scenario_3_years_minus.weekly_i_series, scenario_3_years_minus.name)
plottings.plot_compare_scenario_to_actual_case_for_age_groups(scenario_3_years_minus.I, 0, 4, scenario_3_years_minus.name)
plottings.plot_compare_scenario_to_actual_case_for_age_groups(scenario_3_years_minus.I, 5, 9, scenario_3_years_minus.name)
plottings.plot_compare_scenario_to_actual_case_for_age_groups(scenario_3_years_minus.I, 10, 12, scenario_3_years_minus.name)

## 14. Start vaccination 5 years earlier (2014.09.01)

In [ ]:
scenario_5_years_minus: Scenario = scenarioCalculator.calculate_number_of_cases_for_scenario(Scenarios.STARTING_5_YEAR_MINUS, 1, -5)
ALL_SCENARIOS.append(scenario_5_years_minus)

In [ ]:
plottings.plot_model_actual_scat_comparison(scenario_5_years_minus)
plottings.plot_model_results_age_range_in_range(scenario_5_years_minus.I, 0, 4)
plottings.plot_model_results_age_range_in_range(scenario_5_years_minus.I, 5, 9)
plottings.plot_model_results_age_range_in_range(scenario_5_years_minus.I, 10, 12)

In [ ]:
plottings.plot_compare_scenario_to_actual_case(scenario_5_years_minus.weekly_i_series, scenario_5_years_minus.name)
plottings.plot_compare_scenario_to_actual_case_for_age_groups(scenario_5_years_minus.I, 0, 4, scenario_5_years_minus.name)
plottings.plot_compare_scenario_to_actual_case_for_age_groups(scenario_5_years_minus.I, 5, 9, scenario_5_years_minus.name)
plottings.plot_compare_scenario_to_actual_case_for_age_groups(scenario_5_years_minus.I, 10, 12, scenario_5_years_minus.name)

## 15. Starting the vaccination 8 years earlier (2011.09.01)

In [ ]:
scenario_8_years_minus: Scenario = scenarioCalculator.calculate_number_of_cases_for_scenario(Scenarios.STARTING_8_YEAR_MINUS, 1, -8)
ALL_SCENARIOS.append(scenario_8_years_minus)

In [ ]:
plottings.plot_model_actual_scat_comparison(scenario_8_years_minus)
plottings.plot_model_results_age_range_in_range(scenario_8_years_minus.I, 0, 4)
plottings.plot_model_results_age_range_in_range(scenario_8_years_minus.I, 5, 9)
plottings.plot_model_results_age_range_in_range(scenario_8_years_minus.I, 10, 12)

In [ ]:
plottings.plot_compare_scenario_to_actual_case(scenario_8_years_minus.weekly_i_series, scenario_8_years_minus.name)
plottings.plot_compare_scenario_to_actual_case_for_age_groups(scenario_8_years_minus.I, 0, 4, scenario_8_years_minus.name)
plottings.plot_compare_scenario_to_actual_case_for_age_groups(scenario_8_years_minus.I, 5, 9, scenario_8_years_minus.name)
plottings.plot_compare_scenario_to_actual_case_for_age_groups(scenario_8_years_minus.I, 10, 12, scenario_8_years_minus.name)

## 16. Starting the vaccination 13 years earlier (2006.09.01)

In [ ]:
scenario_13_years_minus: Scenario = scenarioCalculator.calculate_number_of_cases_for_scenario(Scenarios.STARTING_13_YEAR_MINUS, 1, -13)
ALL_SCENARIOS.append(scenario_13_years_minus)

In [ ]:
plottings.plot_model_actual_scat_comparison(scenario_13_years_minus)
plottings.plot_model_results_age_range_in_range(scenario_13_years_minus.I, 0, 4)
plottings.plot_model_results_age_range_in_range(scenario_13_years_minus.I, 5, 9)
plottings.plot_model_results_age_range_in_range(scenario_13_years_minus.I, 10, 12)

In [ ]:
plottings.plot_model_results_age_range_in_range(scenario_13_years_minus.S, 0, 4)
plottings.plot_model_results_age_range_in_range(scenario_13_years_minus.S, 5, 9)
plottings.plot_model_results_age_range_in_range(scenario_13_years_minus.S, 10, 12)

In [ ]:
plottings.plot_compare_scenario_to_actual_case(scenario_13_years_minus.weekly_i_series, scenario_13_years_minus.name)
plottings.plot_compare_scenario_to_actual_case_for_age_groups(scenario_13_years_minus.I, 0, 4, scenario_13_years_minus.name)
plottings.plot_compare_scenario_to_actual_case_for_age_groups(scenario_13_years_minus.I, 5, 9, scenario_13_years_minus.name)
plottings.plot_compare_scenario_to_actual_case_for_age_groups(scenario_13_years_minus.I, 10, 12, scenario_13_years_minus.name)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

i_cum_minus_13 = scenario_13_years_minus.I.sum(axis=0).cumsum()
v_cum_minus_13 = scenario_13_years_minus.V1.sum(axis=0).cumsum()

# ax.plot(state.WEEKLY_INDEX, S_cum, label="Susceptible (cum)", color="blue")
ax.plot(state.WEEKLY_INDEX[state.WEEKLY_INDEX < pd.Timestamp(2012,1,1)], i_cum_minus_13[state.WEEKLY_INDEX < pd.Timestamp(2012, 1, 1)], label="Infected (cum)", color="red")
ax.plot(state.WEEKLY_INDEX[state.WEEKLY_INDEX < pd.Timestamp(2012,1,1)], v_cum_minus_13[state.WEEKLY_INDEX < pd.Timestamp(2012, 1, 1)], label="Vaccinated (cum)", color="green")
#ax.plot(state.WEEKLY_INDEX[state.WEEKLY_INDEX < pd.Timestamp(2007,1,1)], I_cum[state.WEEKLY_INDEX < pd.Timestamp(2007,1,1)] + V_cum[state.WEEKLY_INDEX < pd.Timestamp(2007,1,1)], label="Vaccinated (cum)", color="green")

# Éves beosztás
ax.xaxis.set_major_locator(mdates.YearLocator(1))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

ax.grid(alpha=0.3)
ax.set_title(f"Cumulative I and V")
ax.set_xlabel("Date")
ax.set_ylabel("Cumulative count")
ax.legend()
plt.tight_layout()
plt.show()

## 17. Cumulative plots

In [ ]:
mask = np.ones(len(state.AGE_STRUCTURED_DATA_INDEX), dtype=bool)
plottings.plot_cumulative_cases(ALL_SCENARIOS, state.AGE_STRUCTURED_DATA_INDEX, mask)

In [ ]:
# fig, ax = plt.subplots(figsize=(10, 5))
# index_after_2015_1_1 = state.WEEKLY_INDEX[GlobalConfig.AFTER_2015_1_1_MASK]
#
# # ax.plot(weekly_index, S_cum, label="Susceptible (cum)", color="blue")
# ax.plot(index_after_2015_1_1, i_cum_0[GlobalConfig.AFTER_2015_1_1_MASK], label="No vaccination")
# ax.plot(index_after_2015_1_1, i_cum_25[GlobalConfig.AFTER_2015_1_1_MASK], label="Vaccinating only 25%")
# ax.plot(index_after_2015_1_1, i_cum_50[GlobalConfig.AFTER_2015_1_1_MASK], label="Vaccinating only 50%")
# ax.plot(index_after_2015_1_1, i_cum_75[GlobalConfig.AFTER_2015_1_1_MASK], label="Vaccinating only 75%")
# ax.plot(index_after_2015_1_1, i_cum[GlobalConfig.AFTER_2015_1_1_MASK], label="Actual scenario")
# ax.plot(index_after_2015_1_1, i_cum_minus_1[GlobalConfig.AFTER_2015_1_1_MASK], label="Starting vaccination 1 year earlier")
# ax.plot(index_after_2015_1_1, i_cum_minus_2[GlobalConfig.AFTER_2015_1_1_MASK], label="Starting vaccination 2 year earlier")
# ax.plot(index_after_2015_1_1, i_cum_minus_3[GlobalConfig.AFTER_2015_1_1_MASK], label="Starting vaccination 3 year earlier")
# ax.plot(index_after_2015_1_1, i_cum_minus_5[GlobalConfig.AFTER_2015_1_1_MASK], label="Starting vaccination 5 year earlier")
# #ax.plot(weekly_index[weekly_index < pd.Timestamp(2007,1,1)], I_cum[weekly_index < pd.Timestamp(2007,1,1)] + V_cum[weekly_index < pd.Timestamp(2007,1,1)], label="Vaccinated (cum)", color="green")
#
#
#
# # Éves beosztás
# ax.xaxis.set_major_locator(mdates.YearLocator(1))
# ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
#
# ax.grid(alpha=0.3)
# ax.set_title(f"Cumulative I and V")
# ax.set_xlabel("Date")
# ax.set_ylabel("Cumulative count")
# ax.legend()
# ax.fill_between(index_after_2015_1_1, i_cum_0[GlobalConfig.AFTER_2015_1_1_MASK], alpha=0.5)
# ax.fill_between(index_after_2015_1_1, i_cum_25[GlobalConfig.AFTER_2015_1_1_MASK], alpha=0.5)
# ax.fill_between(index_after_2015_1_1, i_cum_50[GlobalConfig.AFTER_2015_1_1_MASK], alpha=0.5)
# ax.fill_between(index_after_2015_1_1, i_cum_75[GlobalConfig.AFTER_2015_1_1_MASK], alpha=0.5)
# ax.fill_between(index_after_2015_1_1, i_cum[GlobalConfig.AFTER_2015_1_1_MASK], alpha=0.5)
# ax.fill_between(index_after_2015_1_1, i_cum_minus_1[GlobalConfig.AFTER_2015_1_1_MASK], alpha=0.5)
# ax.fill_between(index_after_2015_1_1, i_cum_minus_2[GlobalConfig.AFTER_2015_1_1_MASK], alpha=0.5)
# ax.fill_between(index_after_2015_1_1, i_cum_minus_3[GlobalConfig.AFTER_2015_1_1_MASK], alpha=0.5)
# ax.fill_between(index_after_2015_1_1, i_cum_minus_5[GlobalConfig.AFTER_2015_1_1_MASK], alpha=0.5)
# plt.tight_layout()
# plt.show()

In [ ]:
index_after_2016_1_1 = state.WEEKLY_INDEX[state.AFTER_2016_1_1_MASK]
plottings.plot_cumulative_cases(ALL_SCENARIOS, index_after_2016_1_1, state.AFTER_2016_1_1_MASK)

In [ ]:
# fig, ax = plt.subplots(figsize=(10, 5))
# index_after_2016_1_1 = state.WEEKLY_INDEX[GlobalConfig.AFTER_2016_1_1_MASK]
#
# # ax.plot(weekly_index, S_cum, label="Susceptible (cum)", color="blue")
# ax.plot(index_after_2016_1_1, i_cum_0[GlobalConfig.AFTER_2016_1_1_MASK], label="No vaccination")
# ax.plot(index_after_2016_1_1, i_cum_25[GlobalConfig.AFTER_2016_1_1_MASK], label="Vaccinating only 25%")
# ax.plot(index_after_2016_1_1, i_cum_50[GlobalConfig.AFTER_2016_1_1_MASK], label="Vaccinating only 50%")
# ax.plot(index_after_2016_1_1, i_cum_75[GlobalConfig.AFTER_2016_1_1_MASK], label="Vaccinating only 75%")
# ax.plot(index_after_2016_1_1, i_cum[GlobalConfig.AFTER_2016_1_1_MASK], label="Actual scenario")
# ax.plot(index_after_2016_1_1, i_cum_minus_1[GlobalConfig.AFTER_2016_1_1_MASK], label="Starting vaccination 1 year earlier")
# ax.plot(index_after_2016_1_1, i_cum_minus_3[GlobalConfig.AFTER_2016_1_1_MASK], label="Starting vaccination 3 year earlier")
# #ax.plot(weekly_index[weekly_index < pd.Timestamp(2007,1,1)], I_cum[weekly_index < pd.Timestamp(2007,1,1)] + V_cum[weekly_index < pd.Timestamp(2007,1,1)], label="Vaccinated (cum)", color="green")
#
#
#
# # Éves beosztás
# ax.xaxis.set_major_locator(mdates.YearLocator(1))
# ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
#
# ax.grid(alpha=0.3)
# ax.set_title(f"Cumulative I and V")
# ax.set_xlabel("Date")
# ax.set_ylabel("Cumulative count")
# ax.legend()
# ax.fill_between(index_after_2016_1_1, i_cum_0[GlobalConfig.AFTER_2016_1_1_MASK], 1000000, alpha=0.5)
# ax.fill_between(index_after_2016_1_1, i_cum_25[GlobalConfig.AFTER_2016_1_1_MASK], 1000000, alpha=0.5)
# ax.fill_between(index_after_2016_1_1, i_cum_50[GlobalConfig.AFTER_2016_1_1_MASK], 1000000, alpha=0.5)
# ax.fill_between(index_after_2016_1_1, i_cum_75[GlobalConfig.AFTER_2016_1_1_MASK], 1000000, alpha=0.5)
# ax.fill_between(index_after_2016_1_1, i_cum[GlobalConfig.AFTER_2016_1_1_MASK], 1000000, alpha=0.5)
# ax.fill_between(index_after_2016_1_1, i_cum_minus_1[GlobalConfig.AFTER_2016_1_1_MASK], 1000000, alpha=0.5)
# ax.fill_between(index_after_2016_1_1, i_cum_minus_3[GlobalConfig.AFTER_2016_1_1_MASK], 1000000, alpha=0.5)
# plt.tight_layout()
# plt.show()

## 18. Heatmaps

In [ ]:
for scenario in ALL_SCENARIOS:
    df_scenario = pd.DataFrame(scenario.I, index=state.AGE_GROUPS, columns=state.WEEKLY_INDEX)
    plottings.plot_heatmap(df_scenario, "The change of varicella age distribution - " + scenario.name)

In [ ]:
# df_actual = pd.DataFrame(state.I_BASELINE, index=state.AGE_GROUPS, columns=state.WEEKLY_INDEX)
# df_minus_1 = pd.DataFrame(i_scen_year_minus_1, index=state.AGE_GROUPS, columns=state.WEEKLY_INDEX)
# df_minus_2 = pd.DataFrame(i_scen_year_minus_2, index=state.AGE_GROUPS, columns=state.WEEKLY_INDEX)
# df_minus_3 = pd.DataFrame(i_scen_year_minus_3, index=state.AGE_GROUPS, columns=state.WEEKLY_INDEX)
# df_minus_5 = pd.DataFrame(i_scen_year_minus_5, index=state.AGE_GROUPS, columns=state.WEEKLY_INDEX)
# df_minus_8 = pd.DataFrame(i_scen_year_minus_8, index=state.AGE_GROUPS, columns=state.WEEKLY_INDEX)
# df_minus_13 = pd.DataFrame(scenario_13_years_minus.I, index=state.AGE_GROUPS, columns=state.WEEKLY_INDEX)
# df_level75 = pd.DataFrame(i_scen75, index=state.AGE_GROUPS, columns=state.WEEKLY_INDEX)
# df_level50 = pd.DataFrame(i_scen5, index=state.AGE_GROUPS, columns=state.WEEKLY_INDEX)
# df_level25 = pd.DataFrame(i_scen25, index=state.AGE_GROUPS, columns=state.WEEKLY_INDEX)
# df_level0 = pd.DataFrame(i_scen1, index=state.AGE_GROUPS, columns=state.WEEKLY_INDEX)

In [ ]:
# plot_heatmap(df_actual, "The change of varicella age distribution - actual scenario")
# plot_heatmap(df_minus_1, "The change of varicella age distribution - starting vaccination 1 year earlier")
# plot_heatmap(df_minus_2, "The change of varicella age distribution - starting vaccination 2 year earlier")
# plot_heatmap(df_minus_3, "The change of varicella age distribution - starting vaccination 3 year earlier")
# plot_heatmap(df_minus_5, "The change of varicella age distribution - starting vaccination 5 year earlier")
# plot_heatmap(df_minus_8, "The change of varicella age distribution - starting vaccination 8 year earlier")
# plot_heatmap(df_minus_13, "The change of varicella age distribution - starting vaccination 13 year earlier")
# plot_heatmap(df_level75, "The change of varicella age distribution - vaccinating only 75%")
# plot_heatmap(df_level50, "The change of varicella age distribution - vaccinating only 50")
# plot_heatmap(df_level25, "The change of varicella age distribution - vaccinating only 25%")
# plot_heatmap(df_level0, "The change of varicella age distribution - no vaccination")

## 14. Use mean R_t